In [7]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from vascular_superenhancement.utils.path_config import load_path_config

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
DS_FOLDER = "downsampled_full_fov_128x128x64_crop-17.5"

PID = "Tosbeybrof"  # <-- change this to inspect a different patient

In [8]:
OUTPUT_DIR = Path("slice_order_detail_images")
OUTPUT_DIR.mkdir(exist_ok=True)

ds_root = PATIENT_DATA_DIR / PID / "nifti" / DS_FOLDER

MODALITIES = [
    ("3d_cine",          "3d_cine",          "Cine"),
    ("4d_flow_mag",      "4d_flow_mag",      "Mag"),
    ("4d_flow_vx",       "4d_flow_vx",       "Vx"),
    ("4d_flow_vy",       "4d_flow_vy",       "Vy"),
    ("4d_flow_vz",       "4d_flow_vz",       "Vz"),
    ("4d_flow_diff_vx",  "4d_flow_diff_vx",  "Diff Vx"),
    ("4d_flow_diff_vy",  "4d_flow_diff_vy",  "Diff Vy"),
    ("4d_flow_diff_vz",  "4d_flow_diff_vz",  "Diff Vz"),
]

volumes = {}
for subfolder, prefix, label in MODALITIES:
    path = ds_root / subfolder / f"{prefix}_{PID}_frame_00.nii.gz"
    if path.exists():
        volumes[label] = nib.load(str(path)).get_fdata(dtype=np.float32)
        print(f"{label}: {volumes[label].shape}")
    else:
        print(f"{label}: MISSING")

n_slices = next(iter(volumes.values())).shape[2]
print(f"\nTotal slices: {n_slices}")
print(f"Modalities loaded: {list(volumes.keys())}")

Cine: (128, 128, 64)
Mag: (128, 128, 64)
Vx: (128, 128, 64)
Vy: (128, 128, 64)
Vz: (128, 128, 64)
Diff Vx: (128, 128, 64)
Diff Vy: (128, 128, 64)
Diff Vz: (128, 128, 64)

Total slices: 64
Modalities loaded: ['Cine', 'Mag', 'Vx', 'Vy', 'Vz', 'Diff Vx', 'Diff Vy', 'Diff Vz']


In [9]:
SLICES_PER_PAGE = 8
n_pages = int(np.ceil(n_slices / SLICES_PER_PAGE))
mod_labels = list(volumes.keys())
n_rows = len(mod_labels)

for page in range(n_pages):
    s_start = page * SLICES_PER_PAGE
    s_end = min(s_start + SLICES_PER_PAGE, n_slices)
    page_slices = list(range(s_start, s_end))
    n_cols = len(page_slices)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
    if n_cols == 1:
        axes = axes[:, np.newaxis]
    fig.suptitle(f"{PID}  —  slices {s_start}–{s_end - 1}", fontsize=14, fontweight="bold", y=1.01)

    for r, label in enumerate(mod_labels):
        vol = volumes[label]

        is_signed = label in ("Vx", "Vy", "Vz", "Diff Vx", "Diff Vy", "Diff Vz")
        if is_signed:
            vmax = np.percentile(np.abs(vol), 99)
            vmin = -vmax
            cmap = "RdBu_r"
        else:
            vmin = 0
            vmax = np.percentile(vol, 99)
            cmap = "gray"

        for c, si in enumerate(page_slices):
            ax = axes[r, c]
            ax.imshow(vol[:, :, si].T, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(f"z={si}", fontsize=10)
            if c == 0:
                ax.set_ylabel(label, fontsize=10, rotation=0, labelpad=50, va="center")

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{PID}_page{page:02d}_z{s_start:02d}-{s_end-1:02d}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"Page {page+1}/{n_pages}: slices {s_start}–{s_end-1}")

print(f"\nDone. {n_pages} images saved to {OUTPUT_DIR.resolve()}")

Page 1/8: slices 0–7
Page 2/8: slices 8–15
Page 3/8: slices 16–23
Page 4/8: slices 24–31
Page 5/8: slices 32–39
Page 6/8: slices 40–47
Page 7/8: slices 48–55
Page 8/8: slices 56–63

Done. 8 images saved to /home/ayeluru/vascular-superenhancement-4d-flow/notebooks/sandbox/slice_order_detail_images
